# Adversarial OOD Experiments

**Goal**: Test robustness under increasingly challenging corruption strategies.

**Corruption Hierarchy** (easy → hard):
1. Random (baseline) - any entity
2. Type-constrained - same type as original
3. Popularity-matched - similar training frequency
4. Embedding-similar - nearest neighbors in embedding space
5. Relation-plausible - entities that appear with this relation

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
import json
import random

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# Load FB15k-237 (or your dataset)
from src.data.kg_dataset import KGDataset

dataset = KGDataset('fb15k-237')
train_triples = dataset.train
test_triples = dataset.test

n_entities = dataset.n_entities
n_relations = dataset.n_relations

print(f"Entities: {n_entities}")
print(f"Relations: {n_relations}")
print(f"Train: {len(train_triples)}")
print(f"Test: {len(test_triples)}")

## 1. Build Auxiliary Structures

In [ ]:
# Entity frequency
entity_freq = np.zeros(n_entities)
for h, r, t in train_triples:
    entity_freq[h] += 1
    entity_freq[t] += 1

# Coverage matrix
coverage = np.zeros((n_entities, n_relations), dtype=np.float32)
for h, r, t in train_triples:
    coverage[h, r] = 1
    coverage[t, r] = 1

# Relation → valid tails (entities that appear as tails for this relation)
relation_to_tails = defaultdict(set)
for h, r, t in train_triples:
    relation_to_tails[r].add(t)

print(f"Entity freq range: {entity_freq.min()} to {entity_freq.max()}")
print(f"Coverage density: {coverage.mean():.4f}")

In [ ]:
# Infer entity types from relation co-occurrence
# Entities with similar relation signatures likely have similar types

def infer_entity_types(coverage, n_clusters=50):
    """Cluster entities by coverage pattern to infer types."""
    from sklearn.cluster import MiniBatchKMeans
    
    # Use coverage vectors as features
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1024)
    entity_types = kmeans.fit_predict(coverage)
    
    # Build type → entities map
    type_to_entities = defaultdict(list)
    for e, t in enumerate(entity_types):
        type_to_entities[t].append(e)
    
    return entity_types, type_to_entities

entity_types, type_to_entities = infer_entity_types(coverage)
print(f"Inferred {len(type_to_entities)} entity types")
print(f"Largest type: {max(len(v) for v in type_to_entities.values())} entities")
print(f"Smallest type: {min(len(v) for v in type_to_entities.values())} entities")

## 2. Corruption Strategies

In [ ]:
class CorruptionStrategy:
    """Base class for OOD corruption strategies."""
    
    def __init__(self, n_entities):
        self.n_entities = n_entities
    
    def corrupt(self, h, r, t):
        """Return corrupted tail t'."""
        raise NotImplementedError


class RandomCorruption(CorruptionStrategy):
    """Random uniform corruption."""
    
    def corrupt(self, h, r, t):
        t_prime = random.randint(0, self.n_entities - 1)
        while t_prime == t:
            t_prime = random.randint(0, self.n_entities - 1)
        return t_prime


class TypeConstrainedCorruption(CorruptionStrategy):
    """Replace with entity of same type."""
    
    def __init__(self, n_entities, entity_types, type_to_entities):
        super().__init__(n_entities)
        self.entity_types = entity_types
        self.type_to_entities = type_to_entities
    
    def corrupt(self, h, r, t):
        t_type = self.entity_types[t]
        candidates = [e for e in self.type_to_entities[t_type] if e != t]
        if len(candidates) == 0:
            return random.randint(0, self.n_entities - 1)  # Fallback
        return random.choice(candidates)


class PopularityMatchedCorruption(CorruptionStrategy):
    """Replace with entity of similar training frequency."""
    
    def __init__(self, n_entities, entity_freq, tolerance=0.2):
        super().__init__(n_entities)
        self.entity_freq = entity_freq
        self.tolerance = tolerance
        
        # Pre-compute frequency buckets for efficiency
        self.freq_to_entities = defaultdict(list)
        max_freq = int(entity_freq.max())
        for e, f in enumerate(entity_freq):
            bucket = int(f)
            self.freq_to_entities[bucket].append(e)
    
    def corrupt(self, h, r, t):
        t_freq = self.entity_freq[t]
        low = int(t_freq * (1 - self.tolerance))
        high = int(t_freq * (1 + self.tolerance))
        
        candidates = []
        for bucket in range(low, high + 1):
            candidates.extend([e for e in self.freq_to_entities.get(bucket, []) if e != t])
        
        if len(candidates) == 0:
            return random.randint(0, self.n_entities - 1)
        return random.choice(candidates)


class EmbeddingSimilarCorruption(CorruptionStrategy):
    """Replace with nearest neighbor in embedding space."""
    
    def __init__(self, n_entities, entity_embeddings, k=10):
        super().__init__(n_entities)
        self.k = k
        
        # Build nearest neighbor index
        print("Building KNN index...")
        self.nn = NearestNeighbors(n_neighbors=k+1, algorithm='auto', n_jobs=-1)
        self.nn.fit(entity_embeddings)
        
        # Pre-compute all neighbors
        print("Computing all neighbors...")
        _, self.neighbors = self.nn.kneighbors(entity_embeddings)
        print("Done.")
    
    def corrupt(self, h, r, t):
        # Neighbors include self, so exclude first one
        neighbors = self.neighbors[t, 1:]  # Exclude self
        return random.choice(neighbors)


class RelationPlausibleCorruption(CorruptionStrategy):
    """Replace with entity that appears as tail for this relation."""
    
    def __init__(self, n_entities, relation_to_tails):
        super().__init__(n_entities)
        self.relation_to_tails = {r: list(tails) for r, tails in relation_to_tails.items()}
    
    def corrupt(self, h, r, t):
        candidates = [e for e in self.relation_to_tails.get(r, []) if e != t]
        if len(candidates) == 0:
            return random.randint(0, self.n_entities - 1)
        return random.choice(candidates)

## 3. Load Trained Model

In [ ]:
# Load pre-trained GP-KGE model
# (Assume model is saved or train here)

from src.models.gp_kge import GPKGE

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPKGE(n_entities, n_relations, dim=100).to(device)

# Try to load saved model
try:
    model.load_state_dict(torch.load('../outputs/gpkge_fb15k237.pt', map_location=device))
    print("Loaded pre-trained model.")
except:
    print("No pre-trained model found. Training...")
    from src.training.trainer import train_gpkge
    model = train_gpkge(model, train_triples, n_entities, epochs=50, device=device)
    torch.save(model.state_dict(), '../outputs/gpkge_fb15k237.pt')

model.eval()

In [ ]:
# Get entity embeddings for embedding-similar corruption
with torch.no_grad():
    entity_embeddings = model.entity_mean.weight.cpu().numpy()

## 4. Initialize All Strategies

In [ ]:
strategies = {
    'random': RandomCorruption(n_entities),
    'type_constrained': TypeConstrainedCorruption(n_entities, entity_types, type_to_entities),
    'popularity_matched': PopularityMatchedCorruption(n_entities, entity_freq),
    'embedding_similar': EmbeddingSimilarCorruption(n_entities, entity_embeddings, k=10),
    'relation_plausible': RelationPlausibleCorruption(n_entities, relation_to_tails),
}

## 5. Evaluation Function

In [ ]:
def evaluate_corruption_strategy(model, coverage, test_triples, strategy, n_samples=5000, device='cuda'):
    """Evaluate OOD detection under given corruption strategy."""
    model.eval()
    
    # Sample test triples
    if len(test_triples) > n_samples:
        indices = random.sample(range(len(test_triples)), n_samples)
        sampled = [test_triples[i] for i in indices]
    else:
        sampled = test_triples
    
    # Generate ID and OOD
    id_triples = sampled
    ood_triples = [(h, r, strategy.corrupt(h, r, t)) for h, r, t in sampled]
    
    def get_uncertainties(triples):
        heads = torch.tensor([t[0] for t in triples]).to(device)
        relations = torch.tensor([t[1] for t in triples]).to(device)
        tails = torch.tensor([t[2] for t in triples]).to(device)
        
        with torch.no_grad():
            gp_unc = model.get_gp_uncertainty(heads, tails).cpu().numpy()
        
        h_np = heads.cpu().numpy()
        r_np = relations.cpu().numpy()
        t_np = tails.cpu().numpy()
        
        cov_unc = 2 - coverage[h_np, r_np] - coverage[t_np, r_np]
        
        return gp_unc, cov_unc
    
    id_gp, id_cov = get_uncertainties(id_triples)
    ood_gp, ood_cov = get_uncertainties(ood_triples)
    
    # Labels: 0=ID, 1=OOD
    labels = np.concatenate([np.zeros(len(id_gp)), np.ones(len(ood_gp))])
    
    # GP-only
    gp_scores = np.concatenate([id_gp, ood_gp])
    auroc_gp = roc_auc_score(labels, gp_scores)
    
    # Coverage-only
    cov_scores = np.concatenate([id_cov, ood_cov])
    auroc_cov = roc_auc_score(labels, cov_scores)
    
    # CAGP
    gp_norm = gp_scores * cov_scores.mean() / (gp_scores.mean() + 1e-8)
    cagp_scores = 0.5 * gp_norm + 0.5 * cov_scores
    auroc_cagp = roc_auc_score(labels, cagp_scores)
    
    return {
        'GP-only': auroc_gp,
        'Coverage-only': auroc_cov,
        'CAGP': auroc_cagp,
        'synergy': auroc_cagp - max(auroc_gp, auroc_cov)
    }

## 6. Run All Experiments

In [ ]:
results = {}

for name, strategy in strategies.items():
    print(f"\nEvaluating: {name}")
    result = evaluate_corruption_strategy(model, coverage, test_triples, strategy, device=device)
    results[name] = result
    print(f"  GP-only:      {result['GP-only']:.4f}")
    print(f"  Coverage-only: {result['Coverage-only']:.4f}")
    print(f"  CAGP:         {result['CAGP']:.4f}")
    print(f"  Synergy:      {result['synergy']:+.4f}")

## 7. Results Summary

In [ ]:
import pandas as pd

# Create summary table
summary = []
for name, result in results.items():
    summary.append({
        'Corruption': name,
        'GP-only': result['GP-only'],
        'Coverage-only': result['Coverage-only'],
        'CAGP': result['CAGP'],
        'Synergy': result['synergy']
    })

df_summary = pd.DataFrame(summary)
print("\n" + "="*70)
print("ADVERSARIAL OOD RESULTS (FB15k-237)")
print("="*70)
print(df_summary.to_string(index=False))

In [ ]:
# Visualize
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

strategies_order = ['random', 'type_constrained', 'popularity_matched', 'embedding_similar', 'relation_plausible']
x = range(len(strategies_order))
width = 0.25

gp_vals = [results[s]['GP-only'] for s in strategies_order]
cov_vals = [results[s]['Coverage-only'] for s in strategies_order]
cagp_vals = [results[s]['CAGP'] for s in strategies_order]

ax.bar([i - width for i in x], gp_vals, width, label='GP-only', color='#3498db')
ax.bar([i for i in x], cov_vals, width, label='Coverage-only', color='#e74c3c')
ax.bar([i + width for i in x], cagp_vals, width, label='CAGP', color='#2ecc71')

ax.set_xlabel('Corruption Strategy')
ax.set_ylabel('AUROC')
ax.set_title('OOD Detection Under Different Corruption Strategies')
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in strategies_order])
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../figures/adversarial_ood_comparison.pdf', bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
output = {
    'dataset': 'FB15k-237',
    'results': results,
    'strategies': [
        {'name': 'random', 'description': 'Uniform random tail replacement'},
        {'name': 'type_constrained', 'description': 'Replace with same inferred type'},
        {'name': 'popularity_matched', 'description': 'Replace with similar frequency entity'},
        {'name': 'embedding_similar', 'description': 'Replace with k-NN in embedding space'},
        {'name': 'relation_plausible', 'description': 'Replace with entity appearing with same relation'},
    ]
}

with open('../outputs/adversarial_ood_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved to outputs/adversarial_ood_results.json")

## 9. Key Insights

**Expected Pattern**:
- **Random**: All methods do well (easy baseline)
- **Type-constrained**: Coverage drops, CAGP maintains
- **Popularity-matched**: GP drops (can't distinguish by frequency), CAGP maintains
- **Embedding-similar**: Both drop (adversarial attack on embeddings), CAGP still helps via coverage
- **Relation-plausible**: Coverage drops (by design), GP helps, CAGP maintains

**Key Claim**: CAGP is robust across ALL adversarial settings because the two signals fail on different corruptions.